# Explore

> Decorators for marking code maturity in tutorials and exploratory notebooks.

In [ ]:
#| default_exp utils.explore

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Callable, TypeVar, Any, Optional
from functools import wraps
from dataclasses import dataclass, field
import warnings

## Metadata Classes

In [ ]:
#| export
@dataclass
class ExploreMetadata:
    """Metadata for experimental/exploratory code."""
    reason: str = ""
    blocking: list[str] = field(default_factory=list)
    alternatives: list[str] = field(default_factory=list)

In [ ]:
#| export
@dataclass
class ExtractMetadata:
    """Metadata for code ready to be extracted to the main codebase."""
    target: str  # Target notebook path, e.g., "nbs/00_utils/04_nb.ipynb"
    section: str = ""  # Section header to place under
    priority: str = "medium"  # low | medium | high
    depends_on: list[str] = field(default_factory=list)
    breaking: bool = False  # Would this break existing API?

In [ ]:
#| export
@dataclass
class MigratedMetadata:
    """Metadata for code that has been migrated to the main codebase."""
    to: str  # Full import path, e.g., "lib_name.utils.nb.my_function"
    version: str = ""  # Version when migrated
    redirect: bool = True  # Auto-redirect calls to new location

## Decorators

In [ ]:
#| export
F = TypeVar('F', bound=Callable[..., Any])

In [ ]:
#| export
def explore(
    reason: str = "",
    blocking: list[str] | None = None,
    alternatives: list[str] | None = None
) -> Callable[[F], F]:
    """Mark a function as experimental/exploratory.

    This decorator is a no-op at runtime but attaches metadata
    that can be used by tooling to track experimental code.

    Parameters
    ----------
    reason : str
        Why this approach is being explored.
    blocking : list[str], optional
        What's blocking progress on this exploration.
    alternatives : list[str], optional
        Other approaches being considered.

    Returns
    -------
    Callable
        Decorated function with attached metadata.

    Examples
    --------
    >>> @explore(reason="Testing new algorithm")
    ... def experimental_func():
    ...     return "test"
    >>> experimental_func()
    'test'
    >>> experimental_func._explore_meta.reason
    'Testing new algorithm'
    """
    def decorator(fn: F) -> F:
        fn._explore_meta = ExploreMetadata(
            reason=reason,
            blocking=blocking or [],
            alternatives=alternatives or []
        )
        fn._maturity = "explore"
        return fn
    return decorator

In [ ]:
#| export
def extract(
    target: str,
    section: str = "",
    priority: str = "medium",
    depends_on: list[str] | None = None,
    breaking: bool = False
) -> Callable[[F], F]:
    """Mark a function as ready for extraction to the main codebase.

    This decorator is a no-op at runtime but attaches metadata
    that indicates the function should be moved to `nbs/`.

    Parameters
    ----------
    target : str
        Target notebook path, e.g., "nbs/00_utils/04_nb.ipynb".
    section : str, optional
        Section header to place the function under.
    priority : str, default="medium"
        Extraction priority: "low", "medium", or "high".
    depends_on : list[str], optional
        Package dependencies needed by this function.
    breaking : bool, default=False
        Whether extracting this would break existing API.

    Returns
    -------
    Callable
        Decorated function with attached metadata.

    Examples
    --------
    >>> @extract(target="nbs/00_utils/04_nb.ipynb", priority="high")
    ... def ready_func():
    ...     return "ready"
    >>> ready_func()
    'ready'
    >>> ready_func._extract_meta.target
    'nbs/00_utils/04_nb.ipynb'
    >>> ready_func._extract_meta.priority
    'high'
    """
    def decorator(fn: F) -> F:
        fn._extract_meta = ExtractMetadata(
            target=target,
            section=section,
            priority=priority,
            depends_on=depends_on or [],
            breaking=breaking
        )
        fn._maturity = "extract"
        return fn
    return decorator

In [ ]:
#| export
def migrated(
    to: str,
    version: str = "",
    redirect: bool = True
) -> Callable[[F], F]:
    """Mark a function as migrated to the main codebase.

    When `redirect=True`, calls to the decorated function will
    be automatically redirected to the new location with a
    deprecation warning.

    Parameters
    ----------
    to : str
        Full import path to the new location,
        e.g., "nbdev_mcp.utils.nb.my_function".
    version : str, optional
        Version when the migration occurred.
    redirect : bool, default=True
        If True, automatically redirect calls to new location.

    Returns
    -------
    Callable
        Decorated function that redirects to new location.

    Examples
    --------
    >>> @migrated(to="nbdev_mcp.utils.paths.expand", redirect=False)
    ... def old_expand(path):
    ...     return str(path)
    >>> old_expand._migrated_meta.to
    'nbdev_mcp.utils.paths.expand'
    """
    def decorator(fn: F) -> F:
        fn._migrated_meta = MigratedMetadata(
            to=to,
            version=version,
            redirect=redirect
        )
        fn._maturity = "migrated"

        if redirect:
            @wraps(fn)
            def wrapper(*args, **kwargs):
                # Import the new function dynamically
                parts = to.rsplit('.', 1)
                if len(parts) == 2:
                    module_path, func_name = parts
                    try:
                        import importlib
                        module = importlib.import_module(module_path)
                        new_fn = getattr(module, func_name)
                        warnings.warn(
                            f"{fn.__name__} has been migrated to {to}. "
                            f"Please update your imports.",
                            DeprecationWarning,
                            stacklevel=2
                        )
                        return new_fn(*args, **kwargs)
                    except (ImportError, AttributeError):
                        # Fall back to original if import fails
                        pass
                return fn(*args, **kwargs)

            # Preserve metadata on wrapper
            wrapper._migrated_meta = fn._migrated_meta
            wrapper._maturity = fn._maturity
            return wrapper

        return fn
    return decorator

## Discovery Utilities

In [ ]:
#| export
def get_maturity(fn: Callable) -> str | None:
    """Get the maturity level of a decorated function.

    Parameters
    ----------
    fn : Callable
        Function to check.

    Returns
    -------
    str | None
        One of "explore", "extract", "migrated", or None.

    Examples
    --------
    >>> @explore(reason="test")
    ... def f(): pass
    >>> get_maturity(f)
    'explore'
    >>> def g(): pass
    >>> get_maturity(g) is None
    True
    """
    return getattr(fn, '_maturity', None)

In [ ]:
#| export
def get_extract_targets(module) -> list[tuple[str, str, str]]:
    """Find all functions marked for extraction in a module.

    Parameters
    ----------
    module : module
        Python module to scan.

    Returns
    -------
    list[tuple[str, str, str]]
        List of (function_name, target_notebook, priority) tuples.

    Examples
    --------
    >>> import types
    >>> m = types.ModuleType('test')
    >>> @extract(target="nbs/test.ipynb", priority="high")
    ... def test_fn(): pass
    >>> m.test_fn = test_fn
    >>> get_extract_targets(m)
    [('test_fn', 'nbs/test.ipynb', 'high')]
    """
    results = []
    for name in dir(module):
        obj = getattr(module, name, None)
        if callable(obj) and hasattr(obj, '_extract_meta'):
            meta = obj._extract_meta
            results.append((name, meta.target, meta.priority))
    return results

In [ ]:
#| export
def get_migrated_functions(module) -> list[tuple[str, str]]:
    """Find all functions marked as migrated in a module.

    Parameters
    ----------
    module : module
        Python module to scan.

    Returns
    -------
    list[tuple[str, str]]
        List of (function_name, new_location) tuples.
    """
    results = []
    for name in dir(module):
        obj = getattr(module, name, None)
        if callable(obj) and hasattr(obj, '_migrated_meta'):
            meta = obj._migrated_meta
            results.append((name, meta.to))
    return results

## Tests

In [ ]:
# Test @explore decorator
@explore(reason="Testing algorithm", blocking=["need data"])
def test_explore_fn():
    return "explored"

assert test_explore_fn() == "explored"
assert test_explore_fn._maturity == "explore"
assert test_explore_fn._explore_meta.reason == "Testing algorithm"
assert test_explore_fn._explore_meta.blocking == ["need data"]

In [ ]:
# Test @extract decorator
@extract(
    target="nbs/00_utils/04_nb.ipynb",
    section="## Cell Utilities",
    priority="high",
    depends_on=["nbdev_mcp.utils.paths"]
)
def test_extract_fn(x: int) -> int:
    return x * 2

assert test_extract_fn(5) == 10
assert test_extract_fn._maturity == "extract"
assert test_extract_fn._extract_meta.target == "nbs/00_utils/04_nb.ipynb"
assert test_extract_fn._extract_meta.priority == "high"
assert test_extract_fn._extract_meta.section == "## Cell Utilities"

In [ ]:
# Test @migrated decorator (no redirect)
@migrated(to="nbdev_mcp.utils.paths.expand", version="0.2.0", redirect=False)
def test_migrated_fn(path):
    return str(path)

assert test_migrated_fn("/test") == "/test"
assert test_migrated_fn._maturity == "migrated"
assert test_migrated_fn._migrated_meta.to == "nbdev_mcp.utils.paths.expand"
assert test_migrated_fn._migrated_meta.version == "0.2.0"

In [ ]:
# Test get_maturity utility
assert get_maturity(test_explore_fn) == "explore"
assert get_maturity(test_extract_fn) == "extract"
assert get_maturity(test_migrated_fn) == "migrated"
assert get_maturity(lambda: None) is None

In [ ]:
#| eval: false
# Test get_extract_targets
import types
test_module = types.ModuleType('test_module')
test_module.test_extract_fn = test_extract_fn
test_module.test_explore_fn = test_explore_fn

targets = get_extract_targets(test_module)
assert len(targets) == 1
assert targets[0] == ('test_extract_fn', 'nbs/00_utils/04_nb.ipynb', 'high')

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()